# Instruction tuning under LoRA

Why fit() runs out of memory on a model that generates fine, what LoRA does about it, and what instruction tuning does and does not change.

**Runs on:** GPU with 16 GB — about 30 minutes &nbsp;·&nbsp; **Slides:** [Chapter 16 — Text Generation](../../../course-web-slides/ch16/index.html) &nbsp;·&nbsp; **Section:** 04 — Instruction fine-tuning and LoRA

---

## The dataset

In [ ]:
import json
import keras
import tensorflow as tf

PROMPT_TEMPLATE = "[instruction]\n{}[end]\n[response]\n"
RESPONSE_TEMPLATE = "{}[end]"

dataset_path = keras.utils.get_file(
    origin=("https://hf.co/datasets/databricks/databricks-dolly-15k/"
            "resolve/main/databricks-dolly-15k.jsonl"))

data = {"prompts": [], "responses": []}
with open(dataset_path) as file:
    for line in file:
        features = json.loads(line)
        if features["context"]:
            continue                     # the RAG-shaped examples; skip for now
        data["prompts"].append(PROMPT_TEMPLATE.format(features["instruction"]))
        data["responses"].append(RESPONSE_TEMPLATE.format(features["response"]))

print(f"{len(data['prompts']):,} pairs")
print(repr(data["prompts"][0]))
print(repr(data["responses"][0]))

In [ ]:
ds = tf.data.Dataset.from_tensor_slices(data).shuffle(2000).batch(2)
val_ds = ds.take(100)
train_ds = ds.skip(100)
print("batch size 2 -- that number is about to matter")

## Why fit() would run out of memory

In [ ]:
import keras_hub
gemma_lm = keras_hub.models.CausalLM.from_preset("gemma3_1b", dtype="float32")

params = gemma_lm.count_params()
weights_gb = params * 4 / 1e9
adam_gb = params * 4 * 3 / 1e9      # gradient, velocity, momentum

print(f"parameters:            {params:,}")
print(f"weights (float32):     {weights_gb:.1f} GB")
print(f"Adam optimizer state:  {adam_gb:.1f} GB")
print(f"forward-pass activations: a few GB")
print(f"{'':24s} {'-'*12}")
print(f"total:                 > 16 GB")
print()
print("The model loaded and generated fine, because generation needs")
print("only the weights. Training needs four times that, before")
print("activations. This is the common shape of LLM work: GPU")
print("throughput is a SECONDARY concern to fitting in memory at all.")

## LoRA, from first principles

In [ ]:
from keras import ops
import numpy as np

class Linear(keras.Layer):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.kernel = self.add_weight(shape=(input_dim, output_dim))

    def call(self, inputs):
        return ops.matmul(inputs, self.kernel)


class LoraLinear(keras.Layer):
    def __init__(self, input_dim, output_dim, rank):
        super().__init__()
        self.kernel = self.add_weight(shape=(input_dim, output_dim),
                                      trainable=False)
        self.alpha = self.add_weight(shape=(input_dim, rank))
        self.beta = self.add_weight(shape=(rank, output_dim))

    def call(self, inputs):
        frozen = ops.matmul(inputs, self.kernel)
        update = ops.matmul(ops.matmul(inputs, self.alpha), self.beta)
        return frozen + update

d, r = 2048, 8
print(f"kernel {d}x{d}:         {d*d:>10,} frozen")
print(f"alpha {d}x{r} + beta {r}x{d}: {2*d*r:>10,} trainable")
print(f"{d*d / (2*d*r):.0f}x fewer")

The update does **not** have the expressive power of the original kernel — at the narrow middle the whole update passes through eight floats.

That is the bet, and it holds: **during fine-tuning you no longer need the expressive power you needed during pretraining.** The representation is already built; you are only steering it.

## Turning it on

In [ ]:
gemma_lm.backbone.enable_lora(rank=8)
gemma_lm.summary()

Expected output:

```
 Total params:         1,001,190,528 (3.73 GB)
 Trainable params:         1,304,576 (4.98 MB)
 Non-trainable params:   999,885,952 (3.72 GB)
```

Weights still occupy 3.7 GB; **trainable** parameters are now 5 MB, which takes the optimizer state from gigabytes to megabytes.

Note that the total went *up* slightly — alpha and beta are new weights. **LoRA adds parameters to the model and removes them from the optimizer**, and the optimizer was the problem.

In [ ]:
# The same thing written out, which is what you edit to change the choice.
print("gemma_lm.backbone.trainable = False")
print("for i in range(gemma_lm.backbone.num_layers):")
print("    layer = gemma_lm.backbone.get_layer(f'decoder_block_{i}')")
print("    layer.attention.key_dense.trainable = True")
print("    layer.attention.key_dense.enable_lora(rank=8)")
print("    layer.attention.query_dense.trainable = True")
print("    layer.attention.query_dense.enable_lora(rank=8)")
print()
print("Two decisions the one-liner makes for you: WHICH layers get LoRA,")
print("and at WHAT rank. Adding the value projection, or only the later")
print("blocks, is a one-line change from here.")

## Inside the preprocessor

In [ ]:
preprocessor = gemma_lm.preprocessor
preprocessor.sequence_length = 512
batch = next(iter(train_ds))
x, y, sample_weight = preprocessor(batch)

print("token_ids:   ", x["token_ids"].shape)
print("padding_mask:", x["padding_mask"].shape)
print("y:           ", y.shape)
print("sample_weight:", sample_weight.shape)
print()
print("offset by one, exactly as in chapter 15:")
print(" x:", x["token_ids"][0, :5].numpy())
print(" y:", y[0, :5].numpy())

`sample_weight` restricts the loss to **response tokens**. We do not care about loss on a fixed user prompt, and certainly not on padding.

**Instruction tuning is not a new objective.** It is the pretraining objective, on curated data, with the loss masked to the parts we care about.

## Fine-tuning

In [ ]:
gemma_lm.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.Adam(5e-5),
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()],
)
gemma_lm.fit(train_ds, validation_data=val_ds, epochs=1, verbose=2)

About **55%** next-word accuracy on responses, against 36% for mini-GPT. `weighted_metrics` is what makes the mask reach the metric, so the score is over response tokens only.

## The result

In [ ]:
print(gemma_lm.generate(
    "[instruction]\nWhat is a proper noun?[end]\n[response]\n",
    max_length=512))

It **responds** to the question rather than carrying on the thought of the prompt — and emits `[end]` when finished, because the training data taught it where responses stop.

## The honest test

In [ ]:
print(gemma_lm.generate(
    "[instruction]\nWho is the 542nd president of the United States?[end]\n"
    "[response]\n",
    max_length=512))

**Identical nonsense, now delivered in a helpful tone.**

Instruction tuning changed the *format* of the answer, not its relationship to truth. One thing that helps: train on many pairs where the desired response is *"I don't know"* — which teaches the model to avoid topics where it answers badly. ==A behaviour, not an understanding.==

## Saving just the adapter

In [ ]:
import os

gemma_lm.save("gemma_instruct_lora.keras")
size = os.path.getsize("gemma_instruct_lora.keras") / 1e9
print(f"full model: {size:.2f} GB")
print()
print("In production you would save only the LoRA weights -- a few MB --")
print("and apply them to the base model at load time. That is what makes")
print("it practical to serve dozens of task-specific adapters from ONE")
print("copy of the base model in memory.")

---

## What to take away

- Generation needs the weights; training needs four times that, which is why `fit()` dies on a model that generates fine.
- LoRA freezes the kernel and learns a low-rank correction — a thousandfold cut in trainable parameters.
- Instruction tuning is the pretraining objective on curated data with the loss masked to responses.
- **It changes the form of the answer, not its truth.**